In [1]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://air-quality-api.open-meteo.com/v1/air-quality"

In [2]:
neighborhoods = pd.read_csv('../cairo_neighborhoods.csv')
neighborhoods.head()

,name,name_en,latitude,longitude
0,مصر الجديدة,Heliopolis,30.117722,31.332304
1,عابدين,Abdin,30.044988,31.248863
2,بولاق,Boulak,30.059186,31.234178
3,منشية ناصر,Manshiet Naser,30.036332,31.276329
4,روض الفرج,Rod El-Farag,30.073230,31.236333


In [3]:
def fetch_all_air_qualities(locations_df):
    params = {
        # Pass full lists instead of single floats
        "latitude": locations_df["latitude"].tolist(),
        "longitude": locations_df["longitude"].tolist(),
        "hourly": [
            "pm10",
            "pm2_5",
            "us_aqi",
            "us_aqi_pm2_5",
            "us_aqi_pm10",
            "us_aqi_nitrogen_dioxide",
        ],
        "past_days": 92,
        "domains": "cams_global",
        "timeformat": "unixtime",
    }

    # Single network request for ALL locations
    responses = openmeteo.weather_api(url, params=params)

    data_list = []

    # Iterate over the API responses and location IDs in sync
    for dist_id, response in zip(locations_df["name"], responses):
        hourly = response.Hourly()

        data = {
            "name": dist_id,
            "pm10": hourly.Variables(0).ValuesAsNumpy().mean(),
            "pm2_5": hourly.Variables(1).ValuesAsNumpy().mean(),
            "us_aqi": hourly.Variables(2).ValuesAsNumpy().mean(),
            "us_pm2_5": hourly.Variables(3).ValuesAsNumpy().mean(),
            "us_pm10": hourly.Variables(4).ValuesAsNumpy().mean(),
            "us_no2": hourly.Variables(5).ValuesAsNumpy().mean(),
        }
        data_list.append(data)

    return pd.DataFrame(data_list)

In [9]:
# Chunking example if you have many locations
import time

CHUNK_SIZE = 100
df_list = []

for i in range(0, len(neighborhoods), CHUNK_SIZE):
    try:
        chunk = neighborhoods.iloc[i : i + CHUNK_SIZE]
        df_list.append(fetch_all_air_qualities(chunk))
        print(f'{i}, chuck complement')
        time.sleep(60)
    except Exception as e:
        print("Waiting an Hour")
        time.sleep(3600)

aq_df = pd.concat(df_list, ignore_index=True)

0, chuck complement
100, chuck complement
200, chuck complement
300, chuck complement
400, chuck complement
500, chuck complement
600, chuck complement
700, chuck complement
800, chuck complement
900, chuck complement
1000, chuck complement
1100, chuck complement
1200, chuck complement
1300, chuck complement
1400, chuck complement


In [10]:
aq_df

,name,pm10,pm2_5,us_aqi,us_pm2_5,us_pm10,us_no2
0,مصر الجديدة,36.443260,19.955238,79.361496,70.667221,31.624939,9.333277
1,عابدين,36.443260,19.955238,79.361496,70.667221,31.624939,9.333277
2,بولاق,36.443260,19.955238,79.361496,70.667221,31.624939,9.333277
3,منشية ناصر,36.443260,19.955238,79.361496,70.667221,31.624939,9.333277
4,روض الفرج,36.443260,19.955238,79.361496,70.667221,31.624939,9.333277
...,...,...,...,...,...,...,...
1441,Palm Hills New Cairo Compound,27.691023,16.973969,80.200462,65.024658,24.886189,6.087495
1442,I City New Cairo,27.691023,16.973969,80.200462,65.024658,24.886189,6.087495
1443,Telal East Compound,27.691023,16.973969,80.200462,65.024658,24.886189,6.087495
1444,Trio Gardens Compound,27.691023,16.973969,80.200462,65.024658,24.886189,6.087495


In [11]:
aq_df['name_en'] = neighborhoods[neighborhoods['name'] == aq_df['name']]['name_en']

In [16]:
aq_df['name_en'] = aq_df['name_en'].fillna(aq_df['name'])

In [17]:
aq_df.to_csv("../data/airquality.csv", sep=",", index=False, header=True)